# Liquidity Factor Research

## Research question

Do Amihud illiquidity or rolling dollar volume provide stable cross-sectional return information after costs?

This notebook compares 21- and 63-day liquidity candidates through coverage, robustness, redundancy, factor-shape, and cost-aware tests.

## 1. Setup and data audit

In [2]:
import numpy as np
import pandas as pd

from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.signal_processing import (
    add_sector_neutral_factor,
    process_factor_columns,
)
from alpha_research.validation import calculate_ic_by_horizon

factor_panel = pd.read_parquet(
    PROCESSED_DATA_DIR / "factor_panel.parquet"
)

required_columns = [
    "date",
    "ticker",
    "sector",
    "ret_1d",
    "dollar_volume",
    "forward_ret_1d",
    "forward_ret_5d",
]

missing_columns = [
    column
    for column in required_columns
    if column not in factor_panel.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

factor_panel = factor_panel.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

factor_panel[required_columns].info()

<class 'pandas.DataFrame'>
RangeIndex: 284249 entries, 0 to 284248
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   date            284249 non-null  datetime64[ms]
 1   ticker          284249 non-null  str           
 2   sector          284249 non-null  str           
 3   ret_1d          284148 non-null  float64       
 4   dollar_volume   284249 non-null  float64       
 5   forward_ret_1d  284148 non-null  float64       
 6   forward_ret_5d  283744 non-null  float64       
dtypes: datetime64[ms](1), float64(4), str(2)
memory usage: 20.2 MB


## 2. Candidate construction

In [3]:
liquidity_panel = factor_panel.copy()

valid_dollar_volume = (
    liquidity_panel["dollar_volume"]
    .where(liquidity_panel["dollar_volume"] > 0)
)

# Multiplication by 1e6 only makes the values easier to inspect.
liquidity_panel["daily_amihud_scaled"] = (
    liquidity_panel["ret_1d"].abs()
    / valid_dollar_volume
    * 1_000_000
)

liquidity_panel["amihud_21_raw"] = (
    liquidity_panel
    .groupby("ticker")["daily_amihud_scaled"]
    .transform(
        lambda series: series.rolling(
            window=21,
            min_periods=15,
        ).mean()
    )
)

liquidity_panel["amihud_63_raw"] = (
    liquidity_panel
    .groupby("ticker")["daily_amihud_scaled"]
    .transform(
        lambda series: series.rolling(
            window=63,
            min_periods=42,
        ).mean()
    )
)

liquidity_panel["average_dollar_volume_21"] = (
    liquidity_panel
    .groupby("ticker")["dollar_volume"]
    .transform(
        lambda series: series.rolling(
            window=21,
            min_periods=15,
        ).mean()
    )
)

liquidity_panel["log_dollar_volume_21_raw"] = np.log(
    liquidity_panel["average_dollar_volume_21"]
    .where(
        liquidity_panel["average_dollar_volume_21"] > 0
    )
)

### 2.1 Distribution and coverage

In [4]:
raw_liquidity_columns = [
    "amihud_21_raw",
    "amihud_63_raw",
    "log_dollar_volume_21_raw",
]

liquidity_distribution_summary = (
    liquidity_panel[raw_liquidity_columns]
    .agg(
        [
            "count",
            "mean",
            "std",
            "min",
            "median",
            "max",
            "skew",
        ]
    )
    .T
)

liquidity_distribution_summary["coverage"] = (
    liquidity_panel[raw_liquidity_columns]
    .notna()
    .mean()
)

liquidity_distribution_summary

,count,mean,std,min,median,max,skew,coverage
amihud_21_raw,282736.0,0.000020,0.000040,3.151998e-07,0.000015,0.001858,27.864990,0.994677
amihud_63_raw,280036.0,0.000020,0.000039,4.073037e-07,0.000015,0.001661,26.915064,0.985178
log_dollar_volume_21_raw,282836.0,20.477022,0.905826,1.639434e+01,20.377226,24.829772,1.080242,0.995029


### 2.2 Processing and sector neutralisation

In [5]:
liquidity_factor_map = {
    "amihud_21_raw": "amihud_21",
    "amihud_63_raw": "amihud_63",
    "log_dollar_volume_21_raw": "log_dollar_volume_21",
}

liquidity_panel = process_factor_columns(
    liquidity_panel,
    factor_map=liquidity_factor_map,
    lower_quantile=0.01,
    upper_quantile=0.99,
)

for factor_prefix in liquidity_factor_map.values():
    liquidity_panel = add_sector_neutral_factor(
        liquidity_panel,
        factor_column=f"{factor_prefix}_winsorised",
        output_column=f"{factor_prefix}_sector_neutral_z",
        sector_column="sector",
        min_sector_observations=3,
    )

## 3. Initial predictive screen

In [6]:
liquidity_signals = {
    "Amihud 21-day — raw": "amihud_21_z",
    "Amihud 21-day — sector neutral": (
        "amihud_21_sector_neutral_z"
    ),
    "Amihud 63-day — raw": "amihud_63_z",
    "Amihud 63-day — sector neutral": (
        "amihud_63_sector_neutral_z"
    ),
    "Log dollar volume — raw": "log_dollar_volume_21_z",
    "Log dollar volume — sector neutral": (
        "log_dollar_volume_21_sector_neutral_z"
    ),
}

liquidity_horizon_results = []

for signal_name, factor_column in liquidity_signals.items():
    result = calculate_ic_by_horizon(
        liquidity_panel,
        factor_column=factor_column,
        forward_return_columns=[
            "forward_ret_1d",
            "forward_ret_5d",
        ],
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    liquidity_horizon_results.append(result)

liquidity_horizon_summary = pd.concat(
    liquidity_horizon_results,
    ignore_index=True,
)

liquidity_horizon_summary[
    [
        "signal",
        "forward_return_column",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,forward_return_column,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,Amihud 21-day — raw,forward_ret_1d,2875.0,0.002825,0.163677,0.017259,0.925423,0.508174
1,Amihud 21-day — raw,forward_ret_5d,2871.0,0.008663,0.161443,0.053662,2.875284,0.517938
2,Amihud 21-day — sector neutral,forward_ret_1d,2875.0,0.002469,0.127443,0.019371,1.038649,0.507130
3,Amihud 21-day — sector neutral,forward_ret_5d,2871.0,0.004410,0.124543,0.035406,1.897093,0.508534
4,Amihud 63-day — raw,forward_ret_1d,2848.0,0.002370,0.162061,0.014627,0.780584,0.504213
5,Amihud 63-day — raw,forward_ret_5d,2844.0,0.007669,0.158753,0.048305,2.576075,0.512658
6,Amihud 63-day — sector neutral,forward_ret_1d,2848.0,0.002458,0.126914,0.019367,1.033559,0.502107
7,Amihud 63-day — sector neutral,forward_ret_5d,2844.0,0.004868,0.123491,0.039423,2.102386,0.506329
8,Log dollar volume — raw,forward_ret_1d,2876.0,-0.001510,0.187763,-0.008041,-0.431201,0.503477
9,Log dollar volume — raw,forward_ret_5d,2872.0,0.000280,0.181845,0.001539,0.082466,0.508008


## 4. Robustness diagnostics

In [7]:
from alpha_research.validation import (
    calculate_non_overlapping_ic,
    calculate_quantile_returns,
    calculate_subperiod_ic,
)

amihud_signals = {
    "Amihud 21-day — raw": "amihud_21_z",
    "Amihud 21-day — sector neutral": (
        "amihud_21_sector_neutral_z"
    ),
    "Amihud 63-day — raw": "amihud_63_z",
    "Amihud 63-day — sector neutral": (
        "amihud_63_sector_neutral_z"
    ),
}

### 4.1 Subperiod stability

In [8]:
periods = {
    "2015-2018": ("2015-01-01", "2018-12-31"),
    "2019-2022": ("2019-01-01", "2022-12-31"),
    "2023-present": ("2023-01-01", "2026-12-31"),
}

subperiod_results = []

for signal_name, factor_column in amihud_signals.items():
    result = calculate_subperiod_ic(
        panel=liquidity_panel,
        factor_column=factor_column,
        forward_return_column="forward_ret_5d",
        periods=periods,
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    subperiod_results.append(result)

liquidity_subperiod_summary = pd.concat(
    subperiod_results,
    ignore_index=True,
)

liquidity_subperiod_summary[
    [
        "signal",
        "period",
        "count",
        "mean_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,period,count,mean_ic,ic_ir,t_stat,positive_fraction
0,Amihud 21-day — raw,2015-2018,991.0,0.019230,0.139534,4.392542,0.562059
1,Amihud 21-day — raw,2019-2022,1008.0,0.008927,0.052012,1.651340,0.504960
2,Amihud 21-day — raw,2023-present,872.0,-0.003650,-0.021086,-0.622648,0.482798
3,Amihud 21-day — sector neutral,2015-2018,991.0,0.017026,0.149660,4.711309,0.552977
4,Amihud 21-day — sector neutral,2019-2022,1008.0,0.006467,0.052373,1.662802,0.507937
5,Amihud 21-day — sector neutral,2023-present,872.0,-0.012306,-0.090950,-2.685735,0.458716
6,Amihud 63-day — raw,2015-2018,964.0,0.017446,0.130058,4.038073,0.551867
7,Amihud 63-day — raw,2019-2022,1008.0,0.007593,0.044978,1.427999,0.498016
8,Amihud 63-day — raw,2023-present,872.0,-0.003053,-0.017870,-0.527693,0.486239
9,Amihud 63-day — sector neutral,2015-2018,964.0,0.016275,0.142022,4.409545,0.536307


### 4.2 Non-overlapping IC

In [9]:
non_overlapping_results = []

for signal_name, factor_column in amihud_signals.items():
    result = calculate_non_overlapping_ic(
        panel=liquidity_panel,
        factor_column=factor_column,
        forward_return_column="forward_ret_5d",
        horizon=5,
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    non_overlapping_results.append(result)

liquidity_non_overlapping_ic = pd.concat(
    non_overlapping_results,
    ignore_index=True,
)

liquidity_non_overlapping_summary = (
    liquidity_non_overlapping_ic
    .groupby("signal")
    .agg(
        mean_ic=("mean_ic", "mean"),
        min_ic=("mean_ic", "min"),
        max_ic=("mean_ic", "max"),
        mean_t_stat=("t_stat", "mean"),
        min_positive_fraction=(
            "positive_fraction",
            "min",
        ),
        max_positive_fraction=(
            "positive_fraction",
            "max",
        ),
    )
)

liquidity_non_overlapping_summary

,mean_ic,min_ic,max_ic,mean_t_stat,min_positive_fraction,max_positive_fraction
signal,,,,,,
Amihud 21-day — raw,0.008664,0.006890,0.009760,1.285010,0.493031,0.533101
Amihud 21-day — sector neutral,0.004409,0.003432,0.005973,0.846618,0.500000,0.513937
Amihud 63-day — raw,0.007669,0.006663,0.008956,1.152233,0.495606,0.532513
Amihud 63-day — sector neutral,0.004869,0.004116,0.006357,0.938095,0.490334,0.518453


### 4.3 Redundancy diagnostics

In [10]:
redundancy_pairs = {
    "Raw Amihud vs raw realised vol": (
        "amihud_21_z",
        "realised_vol_63_z",
    ),
    "Sector-neutral Amihud vs sector-neutral realised vol": (
        "amihud_21_sector_neutral_z",
        "realised_vol_63_sector_neutral_z",
    ),
    "Raw Amihud vs log dollar volume": (
        "amihud_21_z",
        "log_dollar_volume_21_z",
    ),
    "Sector-neutral Amihud vs sector-neutral log dollar volume": (
        "amihud_21_sector_neutral_z",
        "log_dollar_volume_21_sector_neutral_z",
    ),
    "Raw Amihud vs beta": (
        "amihud_21_z",
        "beta_126",
    ),
    "Sector-neutral Amihud vs momentum": (
        "amihud_21_sector_neutral_z",
        "mom_12_1m_sector_neutral_z",
    ),
}

correlation_rows = []

for date, group in liquidity_panel.groupby("date"):
    for pair_name, (left_column, right_column) in (
        redundancy_pairs.items()
    ):
        valid = group[
            [left_column, right_column]
        ].dropna()

        if len(valid) < 30:
            continue

        correlation_rows.append(
            {
                "date": date,
                "pair": pair_name,
                "correlation": valid[left_column].corr(
                    valid[right_column],
                    method="spearman",
                ),
            }
        )

daily_liquidity_correlations = pd.DataFrame(
    correlation_rows
)

liquidity_redundancy_summary = (
    daily_liquidity_correlations
    .groupby("pair")["correlation"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

liquidity_redundancy_summary

,count,mean,median,std,min,max
pair,,,,,,
Raw Amihud vs beta,2828,0.027346,0.018687,0.164789,-0.317276,0.502788
Raw Amihud vs log dollar volume,2876,-0.870146,-0.873828,0.037185,-0.962839,-0.764200
Raw Amihud vs raw realised vol,2828,0.106251,0.091866,0.144249,-0.211485,0.485575
Sector-neutral Amihud vs momentum,2639,-0.117394,-0.144052,0.141697,-0.404479,0.311069
Sector-neutral Amihud vs sector-neutral log dollar volume,2876,-0.875339,-0.879992,0.039237,-0.964810,-0.730911
Sector-neutral Amihud vs sector-neutral realised vol,2828,0.159149,0.150583,0.143218,-0.192602,0.551971


### 4.4 Quantile shape

In [11]:
quantile_summary_rows = []

for signal_name, factor_column in amihud_signals.items():
    quantile_returns = calculate_quantile_returns(
        panel=liquidity_panel,
        factor_column=factor_column,
        forward_return_column="forward_ret_5d",
        quantiles=5,
        min_observations=30,
    )

    averages = (
        quantile_returns
        .groupby("quantile")["mean_forward_return"]
        .mean()
    )

    row = {"signal": signal_name}

    for quantile, value in averages.items():
        row[f"Q{quantile}"] = value

    row["Q5_minus_Q1"] = (
        averages.loc[5] - averages.loc[1]
    )
    quantile_summary_rows.append(row)

liquidity_quantile_summary = (
    pd.DataFrame(quantile_summary_rows)
    .set_index("signal")
)

liquidity_quantile_summary

,Q1,Q2,Q3,Q4,Q5,Q5_minus_Q1
signal,,,,,,
Amihud 21-day — raw,0.003984,0.002923,0.003050,0.003091,0.005020,0.001036
Amihud 21-day — sector neutral,0.004035,0.003172,0.003112,0.003601,0.004335,0.000300
Amihud 63-day — raw,0.003901,0.002967,0.003045,0.003296,0.004662,0.000761
Amihud 63-day — sector neutral,0.003993,0.003024,0.003366,0.003565,0.004080,0.000088


## 5. Cost-aware backtests

In [12]:
from alpha_research.backtest import (
    BacktestConfig,
    run_rebalance_offset_backtests,
)

liquidity_backtest_signals = {
    "Amihud 21-day — raw": "amihud_21_z",
    "Amihud 21-day — sector neutral": (
        "amihud_21_sector_neutral_z"
    ),
}

base_config = BacktestConfig(
    rebalance_frequency=5,
    quantiles=5,
    long_quantile=5,
    short_quantile=1,
    long_gross=1.0,
    short_gross=1.0,
    transaction_cost_bps=10.0,
    min_observations=30,
    rebalance_offset=0,
)

liquidity_offset_results = []

for signal_name, factor_column in (
    liquidity_backtest_signals.items()
):
    result = run_rebalance_offset_backtests(
        panel=liquidity_panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        base_config=base_config,
    )

    result["signal"] = signal_name
    liquidity_offset_results.append(result)

liquidity_offset_backtests = pd.concat(
    liquidity_offset_results,
    ignore_index=True,
)

liquidity_offset_backtests[
    [
        "signal",
        "offset",
        "annualised_return",
        "annualised_volatility",
        "sharpe_ratio",
        "max_drawdown",
        "average_rebalance_turnover",
        "total_transaction_cost",
        "maximum_missing_return_weight",
    ]
].sort_values(["signal", "offset"])

,signal,offset,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown,average_rebalance_turnover,total_transaction_cost,maximum_missing_return_weight
0,Amihud 21-day — raw,0,0.026154,0.133956,0.259552,-0.301757,0.372491,0.215300,0.0
1,Amihud 21-day — raw,1,0.032290,0.134347,0.303524,-0.313589,0.370242,0.214000,0.0
2,Amihud 21-day — raw,2,0.024367,0.134150,0.246339,-0.276093,0.375606,0.217100,0.0
3,Amihud 21-day — raw,3,0.018675,0.133188,0.205324,-0.285402,0.374913,0.216700,0.0
4,Amihud 21-day — raw,4,0.029487,0.134213,0.283479,-0.283742,0.365398,0.211200,0.0
5,Amihud 21-day — sector neutral,0,-0.010934,0.104216,-0.053433,-0.339438,0.405545,0.234405,0.0
6,Amihud 21-day — sector neutral,1,-0.005463,0.105218,0.000485,-0.364916,0.411000,0.237558,0.0
7,Amihud 21-day — sector neutral,2,-0.008740,0.104357,-0.031992,-0.354938,0.404289,0.233679,0.0
8,Amihud 21-day — sector neutral,3,-0.004529,0.103593,0.007932,-0.372753,0.405227,0.234221,0.0
9,Amihud 21-day — sector neutral,4,-0.013800,0.105090,-0.079749,-0.375228,0.409415,0.236642,0.0


In [13]:
liquidity_backtest_summary = (
    liquidity_offset_backtests
    .groupby("signal")
    .agg(
        mean_annualised_return=(
            "annualised_return",
            "mean",
        ),
        min_annualised_return=(
            "annualised_return",
            "min",
        ),
        max_annualised_return=(
            "annualised_return",
            "max",
        ),
        mean_sharpe=("sharpe_ratio", "mean"),
        min_sharpe=("sharpe_ratio", "min"),
        max_sharpe=("sharpe_ratio", "max"),
        mean_max_drawdown=("max_drawdown", "mean"),
        mean_rebalance_turnover=(
            "average_rebalance_turnover",
            "mean",
        ),
    )
)

liquidity_backtest_summary

,mean_annualised_return,min_annualised_return,max_annualised_return,mean_sharpe,min_sharpe,max_sharpe,mean_max_drawdown,mean_rebalance_turnover
signal,,,,,,,,
Amihud 21-day — raw,0.026195,0.018675,0.032290,0.259644,0.205324,0.303524,-0.292117,0.371730
Amihud 21-day — sector neutral,-0.008693,-0.013800,-0.004529,-0.031351,-0.079749,0.007932,-0.361455,0.407095


### 5.1 Subperiod stability after costs

In [14]:
liquidity_subperiod_backtests = []

backtest_periods = {
    "2015-2018": ("2015-01-01", "2018-12-31"),
    "2019-2022": ("2019-01-01", "2022-12-31"),
    "2023-present": ("2023-01-01", "2026-12-31"),
}

for period, (start_date, end_date) in backtest_periods.items():
    period_panel = liquidity_panel.loc[
        liquidity_panel["date"].between(
            pd.Timestamp(start_date),
            pd.Timestamp(end_date),
        )
    ].copy()

    result = run_rebalance_offset_backtests(
        panel=period_panel,
        factor_column="amihud_21_z",
        return_column="forward_ret_1d",
        base_config=base_config,
    )

    result["period"] = period
    liquidity_subperiod_backtests.append(result)

liquidity_subperiod_backtests = pd.concat(
    liquidity_subperiod_backtests,
    ignore_index=True,
)

liquidity_subperiod_backtest_summary = (
    liquidity_subperiod_backtests
    .groupby("period")
    .agg(
        mean_annualised_return=("annualised_return", "mean"),
        min_annualised_return=("annualised_return", "min"),
        max_annualised_return=("annualised_return", "max"),
        mean_sharpe=("sharpe_ratio", "mean"),
        min_sharpe=("sharpe_ratio", "min"),
        max_sharpe=("sharpe_ratio", "max"),
        mean_max_drawdown=("max_drawdown", "mean"),
        mean_rebalance_turnover=(
            "average_rebalance_turnover",
            "mean",
        ),
    )
)

liquidity_subperiod_backtest_summary

,mean_annualised_return,min_annualised_return,max_annualised_return,mean_sharpe,min_sharpe,max_sharpe,mean_max_drawdown,mean_rebalance_turnover
period,,,,,,,,
2015-2018,0.094827,0.074764,0.113188,1.012600,0.822066,1.187917,-0.147534,0.401590
2019-2022,0.032932,0.020658,0.044723,0.288498,0.209957,0.362460,-0.292117,0.365865
2023-present,-0.057128,-0.062708,-0.048121,-0.317583,-0.353267,-0.252862,-0.249927,0.362906


## 6. Conclusion

This experiment evaluated Amihud illiquidity and rolling dollar volume as potential liquidity factors in the S&P 100 universe.

### 6.1 Main findings

- **High data coverage:** All candidate measures achieved approximately 98.5%–99.5% coverage.
- **Strong skewness:** Raw Amihud illiquidity was extremely right-skewed, confirming the need for cross-sectional winsorisation.
- **Modest historical predictiveness:** The raw 21-day Amihud signal produced a mean 5-day IC of **0.87%**. Its non-overlapping IC remained positive across all five offsets, ranging from **0.69% to 0.98%**, although the average offset-level \(t\)-statistic was only **1.29**.
- **Weak factor shape:** Quintile returns were not monotonic. Both liquidity extremes sometimes outperformed the middle, and the raw Q5–Q1 spread was only **0.10% over five days**.
- **Distinct from volatility and beta:** Raw Amihud had low average cross-sectional correlations with realised volatility (**0.11**) and beta (**0.03**), so its result was not primarily another volatility or market-exposure effect.
- **Closely related to trading volume:** Its correlation with log dollar volume was approximately **−0.87**, confirming that it largely represents the same underlying liquidity ranking. However, log dollar volume itself had no meaningful predictive IC.
- **Sector neutralisation removed the performance:** The sector-neutral implementation generated a negative mean annualised return of approximately **−0.9%** and no positive rebalance-offset results.

### 6.2 Time stability

The raw 21-day Amihud effect weakened materially over time:

| Period | Mean IC | Mean net annualised return | Mean Sharpe |
|---|---:|---:|---:|
| 2015–2018 | 1.92% | 9.48% | 1.01 |
| 2019–2022 | 0.89% | 3.29% | 0.29 |
| 2023–present | −0.37% | −5.71% | −0.32 |

The recent cost-aware result was negative across every rebalance offset, with annualised returns between **−6.27% and −4.81%**. This confirms that the modest positive full-period backtest was inherited from earlier subperiods and does not represent a persistent current signal.

### 6.3 Research decision

Amihud illiquidity captures an economically meaningful liquidity dimension, but it is **not supported as a production factor** in this universe:

- its predictive relationship is weak and non-monotonic;
- much of its historical performance was concentrated in 2015–2018;
- sector-neutral performance is negative;
- both IC and cost-aware returns reverse during the most recent period.

Therefore:

- do not add Amihud illiquidity to the production factor panel;
- retain this notebook as a documented negative research result;
- do not pursue residualisation against dollar volume at this stage, since the main signal already fails the time-stability test;
- revisit liquidity only if better data or a meaningfully different specification becomes available.

Overall, this experiment illustrates why full-period performance alone is insufficient. Non-overlapping validation, transaction costs, sector controls, factor-shape diagnostics, and subperiod analysis were all necessary to reveal that the apparent historical liquidity premium had decayed.